In [8]:
!pip install transformers datasets torch accelerate scikit-learn
!pip install mlcroissant

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 48.9 MB/s eta 0:00:00
  Created wheel for jsonpath-rw: filename=jsonpath_rw-1.4.0-py3-none-any.whl size=15127 sha256=14d05cfe7094b6967349b14a8bdd99ba76c667465fe99bdf331e299d9c848a09
  Stored in directory: /tmp/pip-ephem-wheel-cache-86ffvf71/wheels/e3/76/6f/c25be6a9e6cc9985b96e8c95997d46790242c6426ef68e754c
Successfully built jsonpath-rw

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [9]:
# ==========================================
# BERT Sentiment Analysis using Hugging Face
# ==========================================

import numpy as np
import torch
from mlcroissant import Dataset
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support



In [10]:
# ==========================================
# Check GPU
# ==========================================

print("=" * 50)
print("PyTorch Version:", torch.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("=" * 50)



PyTorch Version: 2.5.0a0+e000cf0ad9.nv24.10
Using Device: cuda
GPU: NVIDIA H200 MIG 1g.18gb


In [14]:
# ==========================================
# Load Dataset
# ==========================================

print("\nLoading IMDb Dataset...")
dataset = load_dataset("stanfordnlp/imdb")
print(dataset)


#dataset = Dataset(jsonld="https://huggingface.co/api/datasets/stanfordnlp/imdb/croissant")
#records = dataset.records("plain_text")

#print(dataset)


Loading IMDb Dataset...
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [15]:
# ==========================================
# Reduce Dataset (for quick training)
# ==========================================

train_dataset = dataset["train"].shuffle(seed=42).select(range(5000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(1000))



In [16]:
# ==========================================
# Load Tokenizer
# ==========================================

print("\nLoading BERT Tokenizer...")

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")




Loading BERT Tokenizer...


In [17]:
# ==========================================
# Tokenization Function
# ==========================================

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

print("\nTokenizing Dataset...")

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)




Tokenizing Dataset...


Map: 100%|█████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 3210.45 examples/s]


In [18]:
# ==========================================
# Format Dataset
# ==========================================

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)



In [19]:
# ==========================================
# Load BERT Model
# ==========================================

print("\nLoading Pretrained BERT Model...")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)




Loading Pretrained BERT Model...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 16491.47it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were

In [20]:
# ==========================================
# Evaluation Metrics
# ==========================================

def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }



In [21]:
# ==========================================
# Training Arguments
# ==========================================

training_args = TrainingArguments(

    output_dir="./bert_results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_steps=100,

    load_best_model_at_end=True,

    report_to="none"
)



In [22]:
# ==========================================
# Trainer
# ==========================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)



In [23]:
# ==========================================
# Train Model
# ==========================================

print("\nStarting Training...\n")

trainer.train()




Starting Training...



Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.356085,0.381414,0.852000,0.864807,0.825820,0.844864
2,0.224936,0.507823,0.855000,0.835616,0.875000,0.854855


Writing model shards: 100%|███████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.68it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias',

TrainOutput(global_step=1250, training_loss=0.3217440704345703, metrics={'train_runtime': 72.8328, 'train_samples_per_second': 137.301, 'train_steps_per_second': 17.163, 'total_flos': 657777638400000.0, 'train_loss': 0.3217440704345703, 'epoch': 2.0})

In [24]:
# ==========================================
# Evaluate
# ==========================================

print("\nEvaluating Model...\n")

results = trainer.evaluate()

print(results)




Evaluating Model...



Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.224936,0.381712,2,0.854000,0.865385,0.829918,0.847280


{'eval_loss': 0.3817116618156433, 'eval_accuracy': 0.854, 'eval_precision': 0.8653846153846154, 'eval_recall': 0.8299180327868853, 'eval_f1': 0.8472803347280334}


In [25]:
# ==========================================
# Save Model
# ==========================================

print("\nSaving Model...\n")

trainer.save_model("bert_sentiment_model")

tokenizer.save_pretrained("bert_sentiment_model")

print("Model Saved Successfully")




Saving Model...



Writing model shards: 100%|███████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.81it/s]

Model Saved Successfully


In [26]:
# ==========================================
# Load Saved Model
# ==========================================

classifier = pipeline(

    "sentiment-analysis",

    model="bert_sentiment_model",

    tokenizer="bert_sentiment_model",

    device=0 if torch.cuda.is_available() else -1
)



Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 14288.83it/s]


In [27]:
# ==========================================
# Test Custom Sentences
# ==========================================

texts = [

    "This movie is fantastic. I loved it.",

    "Worst movie ever made.",

    "The acting was average.",

    "Excellent direction and amazing screenplay.",

    "I will never watch this again."
]

print("\nPredictions\n")

for sentence in texts:

    prediction = classifier(sentence)[0]

    print("-" * 60)

    print("Sentence :", sentence)

    print("Prediction :", prediction["label"])

    print("Confidence :", round(prediction["score"] * 100, 2), "%")

print("-" * 60)

print("\nCompleted Successfully")


Predictions

------------------------------------------------------------
Sentence : This movie is fantastic. I loved it.
Prediction : LABEL_1
Confidence : 98.36 %
------------------------------------------------------------
Sentence : Worst movie ever made.
Prediction : LABEL_0
Confidence : 97.6 %
------------------------------------------------------------
Sentence : The acting was average.
Prediction : LABEL_0
Confidence : 94.32 %
------------------------------------------------------------
Sentence : Excellent direction and amazing screenplay.
Prediction : LABEL_1
Confidence : 98.15 %
------------------------------------------------------------
Sentence : I will never watch this again.
Prediction : LABEL_0
Confidence : 91.54 %
------------------------------------------------------------

Completed Successfully
